# 기본 설정

라이브러리 import

In [1]:
import pandas as pd
import numpy as np
import os
from pprint import pprint
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import random
import torch
import warnings


# stratify k fold 적용
from sklearn.model_selection import StratifiedKFold
from collections import Counter
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.combine import SMOTEENN, SMOTETomek
from collections import Counter
warnings.filterwarnings("ignore")

랜덤시드 고정

In [2]:
RANDOM_STATE = 736665
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(RANDOM_STATE)

seed_everything(RANDOM_STATE)

## 데이터 불러오기

In [3]:
train = pd.read_csv('./data/train.csv').drop(columns=['ID'])
test = pd.read_csv('./data/test.csv').drop(columns=['ID'])
print(train.shape, test.shape)

(256351, 68) (90067, 67)


# 데이터 전처리

In [4]:
# 한 컬럼에 값이 하나이며 결측치가 없는 열 제거
del_col = []
for col in train.columns:
    if train[col].nunique() == 1:
        if train[col].isnull().sum() == 0:
            print(train[col].value_counts())
            print('결측치 개수 :', train[col].isnull().sum())
            del_col.append(col)
del_col

불임 원인 - 여성 요인
0    256351
Name: count, dtype: int64
결측치 개수 : 0


['불임 원인 - 여성 요인']

In [5]:
# 전부 결측치인 열 제거 
for col in train.columns:
    if train[col].notnull().sum() == 0:
        del_col.append(col)
del_col

['불임 원인 - 여성 요인']

In [6]:
train.drop(columns=del_col, inplace=True)
test.drop(columns=del_col, inplace=True)
train.shape, test.shape

((256351, 67), (90067, 66))

### 결측치 처리

In [7]:
temp = pd.DataFrame(train.isnull().sum(), columns=['결측치 개수'])
temp = temp[temp['결측치 개수'] > 0]
temp['결측치 비율'] = temp['결측치 개수'] / train.shape[0] * 100
temp

,결측치 개수,결측치 비율
임신 시도 또는 마지막 임신 경과 연수,246981,96.344855
특정 시술 유형,2,0.000780
단일 배아 이식 여부,6291,2.454057
착상 전 유전 검사 사용 여부,253633,98.939735
착상 전 유전 진단 사용 여부,6291,2.454057
배아 생성 주요 이유,6291,2.454057
총 생성 배아 수,6291,2.454057
미세주입된 난자 수,6291,2.454057
미세주입에서 생성된 배아 수,6291,2.454057
이식된 배아 수,6291,2.454057


In [8]:
## 결측치 처리
to_zero_columns = []
train[to_zero_columns] = train[to_zero_columns].fillna(0)
test[to_zero_columns] = test[to_zero_columns].fillna(0)


to_minus_columns = ['난자 채취 경과일']
train[to_minus_columns] = train[to_minus_columns].fillna(-1)
test[to_minus_columns] = test[to_minus_columns].fillna(-1)


to_one_columns = []
train[to_one_columns] = train[to_one_columns].fillna(1)
test[to_one_columns] = test[to_one_columns].fillna(1)


to_drop_columns = ['임신 시도 또는 마지막 임신 경과 연수', '난자 해동 경과일', '배아 해동 경과일']
train = train.drop(columns=to_drop_columns)
test = test.drop(columns=to_drop_columns)

to_mean_columns = ['배아 이식 경과일', '난자 혼합 경과일']
train[to_mean_columns] = train[to_mean_columns].fillna(train[to_mean_columns].mean())
test[to_mean_columns] = test[to_mean_columns].fillna(train[to_mean_columns].mean())


train['특정 시술 유형'].fillna(train['특정 시술 유형'].value_counts().index[0], inplace=True)
test['특정 시술 유형'].fillna(train['특정 시술 유형'].value_counts().index[0], inplace=True)

train.isnull().sum().sum(), test.isnull().sum().sum()

(894338, 313512)

In [9]:
# PGD 시술 여부 또는 PGS 시술 여부가 1이면 착상 전 유전 검사 사용 여부도 1로 설정하는 함수
def update_preimplantation_testing(df):
    df.loc[(df["착상 전 유전 진단 사용 여부"] == 1), "PGD 시술 여부"] = 1
    df.loc[(df["PGD 시술 여부"] == 1), "착상 전 유전 진단 사용 여부"] = 1
    df.loc[((df["착상 전 유전 검사 사용 여부"] == 1) & (df["PGD 시술 여부"].isna())), "PGS 시술 여부"] = 1
    return df

# train 및 test 데이터프레임 전처리 적용
train = update_preimplantation_testing(train)
test = update_preimplantation_testing(test)
train.isnull().sum().sum(), test.isnull().sum().sum()

(892558, 312901)

In [10]:
for col in temp[temp['결측치 개수'] == 6291].index:
    if train[col].dtype == 'object':
        train[col].fillna('DI_알 수 없음', inplace=True)
        test[col].fillna('DI_알 수 없음', inplace=True)

    elif train[col].nunique() == 2:
        train[col].fillna(train[col].mode()[0], inplace=True)
        test[col].fillna(train[col].mode()[0], inplace=True)

    else:
        train[col] = train[col].fillna(train[col].mean())
        test[col] = test[col].fillna(train[col].mean())
        
train.isnull().sum().sum(), test.isnull().sum().sum()

(760447, 267205)

In [11]:
temp = pd.DataFrame(train.isnull().sum(), columns=['결측치 개수'])
temp = temp[temp['결측치 개수'] > 0]
temp['결측치 비율'] = temp['결측치 개수'] / train.shape[0] * 100
temp

,결측치 개수,결측치 비율
착상 전 유전 검사 사용 여부,253633,98.939735
PGD 시술 여부,253155,98.753272
PGS 시술 여부,253659,98.949877


In [12]:
to_drop_columns = ['착상 전 유전 검사 사용 여부', 'PGD 시술 여부', 'PGS 시술 여부']
train = train.drop(columns=to_drop_columns)
test = test.drop(columns=to_drop_columns)
train.isnull().sum().sum(), test.isnull().sum().sum()

(0, 0)

### 범주형 변수 인코딩

In [13]:
convert_dict1 = {'0회': 0, '1회': 1, '2회': 2, '3회': 3, '4회': 4, '5회': 5, '6회 이상': 6}

# 변환할 컬럼 리스트
columns_to_convert1 = [
    "총 시술 횟수", "클리닉 내 총 시술 횟수", "IVF 시술 횟수", "DI 시술 횟수",
    "총 임신 횟수", "IVF 임신 횟수", "DI 임신 횟수",
    "총 출산 횟수", "IVF 출산 횟수", "DI 출산 횟수"]

# 변환 딕셔너리 생성
convert_dict2 = {'만18-34세' : 26, '만35-37세' : 36, '만38-39세' : 39, '만40-42세' : 41, '만43-44세' : 44,
                 '만45-50세' : 48, '알 수 없음' : -1}

# 변환할 컬럼 리스트
columns_to_convert2 = ["시술 당시 나이"]

convert_dict3 = {'만20세 이하' : 20, '만21-25세' : 23, '만26-30세' : 28, '만31-35세' : 33,
                 '만36-40세' : 38, '만41-45세' : 43, '알 수 없음' : -1}

# 변환할 컬럼 리스트
columns_to_convert3 = ["난자 기증자 나이", "정자 기증자 나이"]

In [14]:
train[columns_to_convert1] = train[columns_to_convert1].replace(convert_dict1)
test[columns_to_convert1] = test[columns_to_convert1].replace(convert_dict1)

train[columns_to_convert2] = train[columns_to_convert2].replace(convert_dict2)
test[columns_to_convert2] = test[columns_to_convert2].replace(convert_dict2)

train[columns_to_convert3] = train[columns_to_convert3].replace(convert_dict3)
test[columns_to_convert3] = test[columns_to_convert3].replace(convert_dict3)

In [15]:
categorical_columns = train.select_dtypes(include='object').columns
categorical_columns

Index(['시술 시기 코드', '시술 유형', '특정 시술 유형', '배란 유도 유형', '배아 생성 주요 이유', '난자 출처',
       '정자 출처'],
      dtype='object')

In [16]:
from sklearn.preprocessing import OneHotEncoder

onehot_encoder = OneHotEncoder(handle_unknown='ignore')
train_encoded = train.copy()
test_encoded = test.copy()

# 원핫 인코딩 적용
train_encoded_array = onehot_encoder.fit_transform(train[categorical_columns])
test_encoded_array = onehot_encoder.transform(test[categorical_columns])

# 새로운 DataFrame으로 변환
train_encoded_df = pd.DataFrame(train_encoded_array.toarray(), columns=onehot_encoder.get_feature_names_out(categorical_columns), index=train.index) # Changed to toarray()
test_encoded_df = pd.DataFrame(test_encoded_array.toarray(), columns=onehot_encoder.get_feature_names_out(categorical_columns), index=test.index) # Changed to toarray()

# 기존 데이터에서 범주형 컬럼 제거 후 원핫 인코딩된 데이터 추가
train_encoded = train.drop(columns=categorical_columns).join(train_encoded_df)
test_encoded = test.drop(columns=categorical_columns).join(test_encoded_df)
train_encoded.shape, test_encoded.shape

((256351, 112), (90067, 111))

In [17]:
import re
train_encoded.columns = [re.sub(r'[^\w\s]', '_', col) for col in train_encoded.columns]
test_encoded.columns = [re.sub(r'[^\w\s]', '_', col) for col in test_encoded.columns]

# 모델링

### 샘플링 실험

In [18]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# train_data에서 임신 성공 여부을 제외한 모든 열을 feature로 사용
X = train_encoded.drop(columns=["임신 성공 여부"])
y = train_encoded["임신 성공 여부"]

# train 데이터를 5개로 나누어 교차 검증
under_ratio_df = pd.DataFrame(columns=["ratio", 'fold', "model", 'auc'])

ratio = 'default'
for i, index in enumerate(skf.split(X, y)):
    # 데이터 분할
    print(f"{ratio}, Fold {i + 1}")
    X_train, X_val = X.iloc[index[0]], X.iloc[index[1]]
    y_train, y_val = y.iloc[index[0]], y.iloc[index[1]]
    print(Counter(y_train), Counter(y_val))

    print('모델 학습 진행')
    class_counts = Counter(y_train)
    negative_count = class_counts[0]  # 음성 클래스 (보통 0)
    positive_count = class_counts[1]  # 양성 클래스 (보통 1)

    # 클래스 불균형 비율 계산
    class_imbalance_ratio = negative_count / positive_count if positive_count > 0 else 1
    import xgboost as xgb
    import lightgbm as lgb
    import catboost as cat
    xgb = xgb.XGBClassifier(random_state=RANDOM_STATE, scale_pos_weight=class_imbalance_ratio)
    lgbm = lgb.LGBMClassifier(random_state=RANDOM_STATE, is_unbalance=True, verbose=-1)
    cat = cat.CatBoostClassifier(verbose=0, random_state=RANDOM_STATE, auto_class_weights="Balanced")
    for model in [xgb, lgbm]:
        print(model.__class__.__name__)
        model.fit(X_train, y_train)

        y_proba = model.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, y_proba)

        under_ratio_df.loc[len(under_ratio_df), :] = [ratio, i+1, model.__class__.__name__, auc]

for ratio in range(1, 3):
    normal_ratio = ratio

    for i, index in enumerate(skf.split(X, y)):
        # 데이터 분할
        print(f"{ratio}, Fold {i + 1}")
        X_train, X_val = X.iloc[index[0]], X.iloc[index[1]]
        y_train, y_val = y.iloc[index[0]], y.iloc[index[1]]
        print(Counter(y_train), Counter(y_val))

        train = pd.concat([X_train, y_train], axis=1)
        val = pd.concat([X_val, y_val], axis=1)

        df_normal = train[train["임신 성공 여부"] == 0]
        df_abnormal = train[train["임신 성공 여부"] == 1]
        print(f"Total: Normal: {len(df_normal)}, AbNormal: {len(df_abnormal)}")

        df_normal = df_normal.sample(n=int(len(df_abnormal) * normal_ratio), replace=False, random_state=RANDOM_STATE)
        df_concat = pd.concat([df_normal, df_abnormal], axis=0).reset_index(drop=True)
        X_under, y_under = df_concat.drop(columns=["임신 성공 여부"]), df_concat["임신 성공 여부"]

        print("언더샘플링 후 train의 클래스 비율:", Counter(y_under))

        print('모델 학습 진행')
        class_counts = Counter(y_under)
        negative_count = class_counts[0]  # 음성 클래스 (보통 0)
        positive_count = class_counts[1]  # 양성 클래스 (보통 1)

        # 클래스 불균형 비율 계산
        class_imbalance_ratio = negative_count / positive_count if positive_count > 0 else 1
        import xgboost as xgb
        import lightgbm as lgb
        import catboost as cat
        xgb = xgb.XGBClassifier(random_state=RANDOM_STATE, scale_pos_weight=class_imbalance_ratio)
        lgbm = lgb.LGBMClassifier(random_state=RANDOM_STATE, is_unbalance=True, verbose=-1)
        cat = cat.CatBoostClassifier(verbose=0, random_state=RANDOM_STATE, auto_class_weights="Balanced")
        for model in [xgb, lgbm]:
            print(model.__class__.__name__)
            model.fit(X_under, y_under)

            y_proba = model.predict_proba(X_val)[:, 1]
            auc = roc_auc_score(y_val, y_proba)
          
            under_ratio_df.loc[len(under_ratio_df), :] = [ratio, i+1, model.__class__.__name__, auc]


ratio = 'ros'
for i, index in enumerate(skf.split(X, y)):
    # 데이터 분할
    print(f"{ratio}, Fold {i + 1}")
    X_train, X_val = X.iloc[index[0]], X.iloc[index[1]]
    y_train, y_val = y.iloc[index[0]], y.iloc[index[1]]
    print(Counter(y_train), Counter(y_val))

    ros = RandomOverSampler(random_state=RANDOM_STATE)
    X_ros, y_ros = ros.fit_resample(X_train, y_train)
    print("Random Oversampling 클래스 비율:", Counter(y_ros))
    print('모델 학습 진행')
    class_counts = Counter(y_ros)
    negative_count = class_counts[0]  # 음성 클래스 (보통 0)
    positive_count = class_counts[1]  # 양성 클래스 (보통 1)

    # 클래스 불균형 비율 계산
    class_imbalance_ratio = negative_count / positive_count if positive_count > 0 else 1
    import xgboost as xgb
    import lightgbm as lgb
    import catboost as cat
    xgb = xgb.XGBClassifier(random_state=RANDOM_STATE, scale_pos_weight=class_imbalance_ratio)
    lgbm = lgb.LGBMClassifier(random_state=RANDOM_STATE, is_unbalance=True, verbose=-1)
    cat = cat.CatBoostClassifier(verbose=0, random_state=RANDOM_STATE, auto_class_weights="Balanced")

    for model in [xgb, lgbm]:
        print(model.__class__.__name__)
        model.fit(X_ros, y_ros)

        y_proba = model.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, y_proba)

        under_ratio_df.loc[len(under_ratio_df), :] = [ratio, i+1, model.__class__.__name__, auc]


default, Fold 1
Counter({0: 152098, 1: 52982}) Counter({0: 38025, 1: 13246})
모델 학습 진행
XGBClassifier
LGBMClassifier
default, Fold 2
Counter({0: 152098, 1: 52983}) Counter({0: 38025, 1: 13245})
모델 학습 진행
XGBClassifier
LGBMClassifier
default, Fold 3
Counter({0: 152098, 1: 52983}) Counter({0: 38025, 1: 13245})
모델 학습 진행
XGBClassifier
LGBMClassifier
default, Fold 4
Counter({0: 152099, 1: 52982}) Counter({0: 38024, 1: 13246})
모델 학습 진행
XGBClassifier
LGBMClassifier
default, Fold 5
Counter({0: 152099, 1: 52982}) Counter({0: 38024, 1: 13246})
모델 학습 진행
XGBClassifier
LGBMClassifier
1, Fold 1
Counter({0: 152098, 1: 52982}) Counter({0: 38025, 1: 13246})
Total: Normal: 152098, AbNormal: 52982
언더샘플링 후 train의 클래스 비율: Counter({0: 52982, 1: 52982})
모델 학습 진행
XGBClassifier
LGBMClassifier
1, Fold 2
Counter({0: 152098, 1: 52983}) Counter({0: 38025, 1: 13245})
Total: Normal: 152098, AbNormal: 52983
언더샘플링 후 train의 클래스 비율: Counter({0: 52983, 1: 52983})
모델 학습 진행
XGBClassifier
LGBMClassifier
1, Fold 3
Counter({0: 1

In [19]:
under_ratio_df.groupby(["ratio", "model"]).mean()[['auc']]

auc
ratio   model                   
1       LGBMClassifier   0.73778
        XGBClassifier   0.733022
2       LGBMClassifier   0.73852
        XGBClassifier   0.734382
default LGBMClassifier  0.738864
        XGBClassifier   0.735134
ros     LGBMClassifier  0.738255
        XGBClassifier   0.734634

In [20]:
under_ratio_df.groupby(["ratio", "model"]).mean()[['auc']].sort_values(by='auc', ascending=False).head()

,,auc
ratio,model,
default,LGBMClassifier,0.738864
2,LGBMClassifier,0.73852
ros,LGBMClassifier,0.738255
1,LGBMClassifier,0.73778
default,XGBClassifier,0.735134


In [21]:
under_ratio_df.groupby(["ratio"])[['auc']].mean().sort_values(by='auc', ascending=False)

,auc
ratio,
default,0.736999
2,0.736451
ros,0.736444
1,0.735401


In [22]:
top5_df = under_ratio_df.groupby(["ratio", "model"]).mean()[['auc']].sort_values(by='auc', ascending=False).head()
top5_df.reset_index(inplace=True)
top5_df

,ratio,model,auc
0,default,LGBMClassifier,0.738864
1,2,LGBMClassifier,0.73852
2,ros,LGBMClassifier,0.738255
3,1,LGBMClassifier,0.73778
4,default,XGBClassifier,0.735134


#### 하이퍼 파라미터 튜닝

In [23]:
import optuna

# train_data에서 임신 성공 여부을 제외한 모든 열을 feature로 사용
X = train_encoded.drop(columns=["임신 성공 여부"])
y = train_encoded["임신 성공 여부"]

# Optuna objective function
def objective(trial, ratio, model_name):

    # Stratified KFold Cross Validation
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    auc_scores = []
    for i, index in enumerate(skf.split(X, y)):
        X_train, X_val = X.iloc[index[0]], X.iloc[index[1]]
        y_train, y_val = y.iloc[index[0]], y.iloc[index[1]]
        train = pd.concat([X_train, y_train], axis=1)
        val = pd.concat([X_val, y_val], axis=1)
        if ratio == 'default':
            X_train_sample, y_train_sample = X_train, y_train

        elif ratio == 'ros':
            ros = RandomOverSampler(random_state=RANDOM_STATE)
            X_ros, y_ros = ros.fit_resample(X_train, y_train)
            X_train_sample, y_train_sample = X_ros, y_ros 

        elif ratio == 'smote':
            smote = SMOTE(sampling_strategy=0.7, random_state=RANDOM_STATE)
            X_smote, y_smote = smote.fit_resample(X_train, y_train)
            X_train_sample, y_train_sample = X_smote, y_smote

        else:
            normal_ratio = ratio
            df_normal = train[train["임신 성공 여부"] == 0]
            df_abnormal = train[train["임신 성공 여부"] == 1]

            df_normal = df_normal.sample(n=int(len(df_abnormal) * normal_ratio), replace=False, random_state=RANDOM_STATE)
            df_concat = pd.concat([df_normal, df_abnormal], axis=0).reset_index(drop=True)
            X_train_sample, y_train_sample = df_concat.drop(columns=["임신 성공 여부"]), df_concat["임신 성공 여부"]


        if model_name == 'XGBClassifier':
            param = {
                'objective': 'binary:logistic',
                'eval_metric': 'auc',
                'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.1),  # 학습률 범위 좁힘
                'max_depth': trial.suggest_int('max_depth', 3, 7),  # max_depth 범위 좁힘
                'min_child_weight': trial.suggest_loguniform('min_child_weight', 1e-2, 5),  # 최소 가중치 범위 좁힘
                'subsample': trial.suggest_uniform('subsample', 0.6, 0.9),  # 샘플 비율 범위 좁힘
                'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.6, 0.9),  # 특성 비율 범위 좁힘
                'lambda': trial.suggest_loguniform('lambda', 1e-3, 1.0),  # L2 정규화 범위 좁힘
                'alpha': trial.suggest_loguniform('alpha', 1e-3, 1.0),  # L1 정규화 범위 좁힘
                'n_estimators': trial.suggest_int('n_estimators', 500, 5000),
                'scale_pos_weight': trial.suggest_loguniform('scale_pos_weight', 1, 5),  # 불균형 클래스 가중치 범위 좁힘
                'gamma': trial.suggest_uniform('gamma', 0, 3),  # 트리 분할 시 최소 손실 감소값 범위 좁힘
                'max_delta_step': trial.suggest_int('max_delta_step', 0, 5)  # 과적합 방지 범위 좁힘
            }

            model = xgb.XGBClassifier(**param, random_state=RANDOM_STATE, n_jobs=-1)

        elif model_name == 'LGBMClassifier':
            param = {
                'objective': 'binary',
                'metric': 'auc',
                'learning_rate': trial.suggest_loguniform('learning_rate', 0.005, 0.1),  # 학습률 범위 확장
                'num_leaves': trial.suggest_int('num_leaves', 20, 80),  # num_leaves 범위 확장
                'max_depth': trial.suggest_int('max_depth', 3, 10),  # max_depth 범위 확장
                'min_child_samples': trial.suggest_int('min_child_samples', 10, 150),  # 최소 샘플 수 범위 확장
                'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),  # 샘플링 비율 범위 확장
                'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1.0),  # 특성 비율 범위 확장
                'lambda_l1': trial.suggest_loguniform('lambda_l1', 1e-4, 10),  # L1 정규화 범위 확장
                'lambda_l2': trial.suggest_loguniform('lambda_l2', 1e-4, 2.0),  # L2 정규화 범위 확장
                'max_bin': trial.suggest_int('max_bin', 100, 600),  # bin 수 범위 확장
                'n_estimators': trial.suggest_int('n_estimators', 500, 6000),  # n_estimators 범위 확장
                'scale_pos_weight': trial.suggest_loguniform('scale_pos_weight', 0.5, 5),  # 클래스 불균형 조정
                'min_child_weight': trial.suggest_loguniform('min_child_weight', 1e-3, 5),  # 최소 가중치 범위 확장
                'feature_fraction': trial.suggest_uniform('feature_fraction', 0.5, 1.0),  # 특성 비율 범위 확장
                'bagging_fraction': trial.suggest_uniform('bagging_fraction', 0.5, 1.0),  # 배깅 샘플링 비율 범위 확장
                'bagging_freq': trial.suggest_int('bagging_freq', 1, 7)  # 배깅 적용 주기 범위 확장
            }

            # param = {
            #     'objective': 'binary',
            #     'metric': 'auc',
            #     'learning_rate': trial.suggest_loguniform('learning_rate', 0.001, 0.05),  
            #     'num_leaves': trial.suggest_int('num_leaves', 31, 127),  
            #     'max_depth': trial.suggest_int('max_depth', 4, 12),  
            #     'min_child_samples': trial.suggest_int('min_child_samples', 10, 120),  
            #     'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.7, 0.9),  
            #     'lambda_l1': trial.suggest_loguniform('lambda_l1', 1e-3, 5),  
            #     'lambda_l2': trial.suggest_loguniform('lambda_l2', 1e-3, 1.0),  
            #     'max_bin': trial.suggest_int('max_bin', 200, 500),  
            #     'n_estimators': trial.suggest_int('n_estimators', 1000, 8000),  
            #     'scale_pos_weight': trial.suggest_loguniform('scale_pos_weight', 1.0, 3.5),  
            #     'min_child_weight': trial.suggest_loguniform('min_child_weight', 1e-2, 10),  
            #     'feature_fraction': trial.suggest_uniform('feature_fraction', 0.7, 0.9),  
            #     'bagging_fraction': trial.suggest_uniform('bagging_fraction', 0.7, 0.9),  
            #     'bagging_freq': trial.suggest_int('bagging_freq', 1, 7)
            # }


            model = lgb.LGBMClassifier(**param, random_state=RANDOM_STATE, n_jobs=-1)
            

        elif model_name == 'CatBoostClassifier':
            param = {
                'objective': 'Logloss',
                'eval_metric': 'AUC',
                'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.1),  # 학습률 범위 좁힘
                'depth': trial.suggest_int('depth', 3, 7),  # depth 범위 좁힘
                'l2_leaf_reg': trial.suggest_loguniform('l2_leaf_reg', 1e-3, 5.0),  # L2 정규화 범위 좁힘
                'random_strength': trial.suggest_uniform('random_strength', 0.5, 3),  # 잡음 정도 범위 좁힘
                'bagging_temperature': trial.suggest_uniform('bagging_temperature', 0, 0.3),  # 배깅 온도 범위 좁힘
                # 'border_count': trial.suggest_int('border_count', 32, 100),  # border 수 범위 좁힘
                'colsample_bylevel': trial.suggest_uniform('colsample_bylevel', 0.6, 0.9),  # 특성 비율 범위 좁힘
                'iterations': trial.suggest_int('iterations', 500, 2000),
                # 'scale_pos_weight': trial.suggest_loguniform('scale_pos_weight', 1, 5),  # 불균형 클래스 가중치 범위 좁힘
                'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 50),  # 리프 노드 최소 데이터 수 범위 좁힘
                'max_bin': trial.suggest_int('max_bin', 100, 150)  # bin 수 범위 좁힘
            }
            model = cat.CatBoostClassifier(**param, verbose=0, random_state=RANDOM_STATE, auto_class_weights="Balanced")    

        else:
            raise ValueError("Invalid model name")
        model.fit(X_train_sample, y_train_sample)

        # 예측 및 AUC 평가
        y_prob = model.predict_proba(X_val)[:, 1]  # 클래스 1의 확률 예측
        auc = roc_auc_score(y_val, y_prob)
        auc_scores.append(auc)

    # 평균 AUC 반환 (Optuna는 최대화 목표)
    return np.mean(auc_scores)

In [24]:
import optuna

def optimize_ran(ratio, model_name):
    def objective_wrapper(trial):
        return objective(trial, ratio, model_name)
    
    # **초반에는 RandomSampler 사용 (빠른 탐색)**
    study = optuna.create_study(
        direction='maximize', 
        sampler=optuna.samplers.RandomSampler(seed=RANDOM_STATE)  # 랜덤 샘플링 시드 고정
    )
    study.optimize(objective_wrapper, n_trials=20)  # 초반 10회 랜덤 탐색

    print('Best hyperparameters: ', study.best_params, study.best_value)
    return study.best_params, study.best_value, study

params, auc, study = optimize_ran(ratio=top5_df['ratio'][0], model_name=top5_df['model'][0])
top5_df.loc[0, ['param', 'tuned_auc']] =  params, auc

import pickle

# Study 저장
with open("./data/optuna_study.pkl", "wb") as f:
    pickle.dump(study, f)

# Study 불러오기
with open("./data/optuna_study.pkl", "rb") as f:
    study = pickle.load(f)

[I 2025-02-26 09:36:59,394] A new study created in memory with name: no-name-cc14d8a5-b019-42c9-8240-434045a99c48
[I 2025-02-26 09:38:57,646] Trial 0 finished with value: 0.7338829848430737 and parameters: {'learning_rate': 0.015425143772139221, 'num_leaves': 66, 'max_depth': 9, 'min_child_samples': 133, 'subsample': 0.8653901486840079, 'colsample_bytree': 0.9983917544473779, 'lambda_l1': 0.6105295749847629, 'lambda_l2': 0.044168222863989644, 'max_bin': 126, 'n_estimators': 2273, 'scale_pos_weight': 1.95333788496361, 'min_child_weight': 0.05291362037823648, 'feature_fraction': 0.9523240386576735, 'bagging_fraction': 0.6518583351261913, 'bagging_freq': 7}. Best is trial 0 with value: 0.7338829848430737.
[I 2025-02-26 09:40:45,737] Trial 1 finished with value: 0.7333386716012593 and parameters: {'learning_rate': 0.03116375846521187, 'num_leaves': 30, 'max_depth': 10, 'min_child_samples': 56, 'subsample': 0.7221916135181802, 'colsample_bytree': 0.7974033679912669, 'lambda_l1': 0.009015009

Best hyperparameters:  {'learning_rate': 0.009472902001113578, 'num_leaves': 44, 'max_depth': 4, 'min_child_samples': 96, 'subsample': 0.9263304974726281, 'colsample_bytree': 0.9545125683629523, 'lambda_l1': 0.003983286973198198, 'lambda_l2': 0.07408975311080031, 'max_bin': 211, 'n_estimators': 1731, 'scale_pos_weight': 2.160158734295782, 'min_child_weight': 1.7815490358345771, 'feature_fraction': 0.9178770758397096, 'bagging_fraction': 0.5337384100461013, 'bagging_freq': 7} 0.7395980824208157


In [25]:
study.optimize(lambda trial: objective(trial, top5_df['ratio'][0], top5_df['model'][0]), n_trials=30)

params, auc = study.best_params, study.best_value
print(params, auc)
top5_df.loc[0, ['param', 'tuned_auc']] =  params, auc

# Study 저장
with open("./data/optuna_study_랜덤완.pkl", "wb") as f:
    pickle.dump(study, f)

# Study 불러오기
with open("./data/optuna_study_랜덤완.pkl", "rb") as f:
    study = pickle.load(f)

[I 2025-02-26 10:37:35,018] Trial 20 finished with value: 0.733604076408731 and parameters: {'learning_rate': 0.011345626605096769, 'num_leaves': 75, 'max_depth': 9, 'min_child_samples': 46, 'subsample': 0.9159144260138378, 'colsample_bytree': 0.5909900454472763, 'lambda_l1': 0.012880736066555386, 'lambda_l2': 0.0010195141515696757, 'max_bin': 256, 'n_estimators': 3230, 'scale_pos_weight': 3.4058426081316924, 'min_child_weight': 0.043935179376647635, 'feature_fraction': 0.9855540025756568, 'bagging_fraction': 0.9855584519671904, 'bagging_freq': 4}. Best is trial 10 with value: 0.7395980824208157.
[I 2025-02-26 10:42:07,224] Trial 21 finished with value: 0.7082062979721606 and parameters: {'learning_rate': 0.044194419131192854, 'num_leaves': 58, 'max_depth': 10, 'min_child_samples': 35, 'subsample': 0.8471550516588009, 'colsample_bytree': 0.7672541724132067, 'lambda_l1': 0.019368913791820087, 'lambda_l2': 0.00011058211950156807, 'max_bin': 232, 'n_estimators': 5961, 'scale_pos_weight': 

{'learning_rate': 0.005817058550512746, 'num_leaves': 55, 'max_depth': 9, 'min_child_samples': 90, 'subsample': 0.7322451188571679, 'colsample_bytree': 0.5382926852387944, 'lambda_l1': 0.00380243981756742, 'lambda_l2': 0.029629484171191444, 'max_bin': 338, 'n_estimators': 1214, 'scale_pos_weight': 0.7743415462679393, 'min_child_weight': 0.04738493030921971, 'feature_fraction': 0.5893372806238872, 'bagging_fraction': 0.6654501994057781, 'bagging_freq': 6} 0.7398059205596593


In [26]:
study.sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)  # TPESampler 시드 고정
print(study.sampler)

study.optimize(lambda trial: objective(trial, top5_df['ratio'][0], top5_df['model'][0]), n_trials=100)

params, auc = study.best_params, study.best_value
print(params, auc)
top5_df.loc[0, ['param', 'tuned_auc']] =  params, auc

# Study 저장
with open("./data/optuna_study_100.pkl", "wb") as f:
    pickle.dump(study, f)

# Study 불러오기
with open("./data/optuna_study_100.pkl", "rb") as f:
    study = pickle.load(f)

TPESampler


[I 2025-02-26 13:33:20,116] Trial 50 finished with value: 0.7390716789215895 and parameters: {'learning_rate': 0.009400077337916118, 'num_leaves': 28, 'max_depth': 9, 'min_child_samples': 84, 'subsample': 0.8171180821269252, 'colsample_bytree': 0.5508358063588705, 'lambda_l1': 0.004632619736363163, 'lambda_l2': 0.32100657383455367, 'max_bin': 158, 'n_estimators': 540, 'scale_pos_weight': 2.1852904190483646, 'min_child_weight': 0.0031187523232613194, 'feature_fraction': 0.6024817321250162, 'bagging_fraction': 0.5049360999237884, 'bagging_freq': 6}. Best is trial 37 with value: 0.7398059205596593.
[I 2025-02-26 13:34:48,194] Trial 51 finished with value: 0.7396838604068299 and parameters: {'learning_rate': 0.008522646668550087, 'num_leaves': 24, 'max_depth': 9, 'min_child_samples': 24, 'subsample': 0.8781669377132313, 'colsample_bytree': 0.5505961619084916, 'lambda_l1': 0.004322676596353319, 'lambda_l2': 0.009903555783612312, 'max_bin': 511, 'n_estimators': 2055, 'scale_pos_weight': 0.95

{'learning_rate': 0.005034983060822145, 'num_leaves': 36, 'max_depth': 7, 'min_child_samples': 147, 'subsample': 0.9109603413137686, 'colsample_bytree': 0.7468857214614529, 'lambda_l1': 0.0017197720809456513, 'lambda_l2': 0.2756008731148777, 'max_bin': 494, 'n_estimators': 1691, 'scale_pos_weight': 1.7724248071595592, 'min_child_weight': 1.0892590244720923, 'feature_fraction': 0.8154619713418669, 'bagging_fraction': 0.5148378269219118, 'bagging_freq': 2} 0.7400296340017085


In [27]:
study.optimize(lambda trial: objective(trial, top5_df['ratio'][0], top5_df['model'][0]), n_trials=100)

params, auc = study.best_params, study.best_value
print(params, auc)
top5_df.loc[0, ['param', 'tuned_auc']] =  params, auc


# Study 저장
with open("./data/optuna_study_200.pkl", "wb") as f:
    pickle.dump(study, f)

# Study 불러오기
with open("./data/optuna_study_200.pkl", "rb") as f:
    study = pickle.load(f)

[I 2025-02-26 16:38:25,903] Trial 150 finished with value: 0.7399874065119407 and parameters: {'learning_rate': 0.005421446931613538, 'num_leaves': 33, 'max_depth': 8, 'min_child_samples': 124, 'subsample': 0.9556357111815003, 'colsample_bytree': 0.8143559646751638, 'lambda_l1': 0.0011049977580733963, 'lambda_l2': 0.40003315903423925, 'max_bin': 520, 'n_estimators': 1610, 'scale_pos_weight': 3.272190364686397, 'min_child_weight': 2.1952330639339515, 'feature_fraction': 0.7854122208997758, 'bagging_fraction': 0.5269999340339792, 'bagging_freq': 2}. Best is trial 130 with value: 0.7400296340017085.
[I 2025-02-26 16:39:44,075] Trial 151 finished with value: 0.7399573081908659 and parameters: {'learning_rate': 0.005445654428901667, 'num_leaves': 31, 'max_depth': 8, 'min_child_samples': 124, 'subsample': 0.9562710276169445, 'colsample_bytree': 0.8123793376093087, 'lambda_l1': 0.001983638096728173, 'lambda_l2': 0.25990254223760195, 'max_bin': 521, 'n_estimators': 1732, 'scale_pos_weight': 3.

{'learning_rate': 0.006192984010225524, 'num_leaves': 30, 'max_depth': 8, 'min_child_samples': 145, 'subsample': 0.9530598170006493, 'colsample_bytree': 0.7637820484562927, 'lambda_l1': 0.00010612350732401384, 'lambda_l2': 1.9403017534727327, 'max_bin': 555, 'n_estimators': 1746, 'scale_pos_weight': 2.952421231501966, 'min_child_weight': 2.2003992924152467, 'feature_fraction': 0.7373377710707284, 'bagging_fraction': 0.5426901866174296, 'bagging_freq': 2} 0.7400376427126525


In [28]:
study.optimize(lambda trial: objective(trial, top5_df['ratio'][0], top5_df['model'][0]), n_trials=100)

params, auc = study.best_params, study.best_value
print(params, auc)
top5_df.loc[0, ['param', 'tuned_auc']] =  params, auc


# Study 저장
with open("./data/optuna_study_300.pkl", "wb") as f:
    pickle.dump(study, f)

# Study 불러오기
with open("./data/optuna_study_300.pkl", "rb") as f:
    study = pickle.load(f)

[I 2025-02-26 20:14:21,903] Trial 250 finished with value: 0.7397652803202472 and parameters: {'learning_rate': 0.005211623906465605, 'num_leaves': 64, 'max_depth': 8, 'min_child_samples': 150, 'subsample': 0.9904629382061906, 'colsample_bytree': 0.8837588823478838, 'lambda_l1': 0.00033387480482500434, 'lambda_l2': 1.6203961252289103, 'max_bin': 600, 'n_estimators': 1910, 'scale_pos_weight': 3.564629600125515, 'min_child_weight': 4.751489866363486, 'feature_fraction': 0.8106257988650707, 'bagging_fraction': 0.5104587866282918, 'bagging_freq': 2}. Best is trial 200 with value: 0.7400376427126525.
[I 2025-02-26 20:15:48,348] Trial 251 finished with value: 0.7399784665137094 and parameters: {'learning_rate': 0.0050564095549366665, 'num_leaves': 34, 'max_depth': 8, 'min_child_samples': 150, 'subsample': 0.9755805032123083, 'colsample_bytree': 0.8809672555389535, 'lambda_l1': 0.0003713641103182995, 'lambda_l2': 0.0013605023438443985, 'max_bin': 592, 'n_estimators': 1828, 'scale_pos_weight':

{'learning_rate': 0.005404665080038044, 'num_leaves': 37, 'max_depth': 8, 'min_child_samples': 142, 'subsample': 0.6651803742719826, 'colsample_bytree': 0.7685478000166595, 'lambda_l1': 2.081721942205924, 'lambda_l2': 1.9845698568428882, 'max_bin': 552, 'n_estimators': 1692, 'scale_pos_weight': 3.574385124249283, 'min_child_weight': 2.9798133779257356, 'feature_fraction': 0.8194483752728194, 'bagging_fraction': 0.5203599421398333, 'bagging_freq': 2} 0.740060511984433


In [30]:
optuna.visualization.plot_param_importances(study) # 파라미터 중요도 확인 그래프

In [31]:
optuna.visualization.plot_optimization_history(study) # 최적화 과정 시각화

In [38]:
optuna_resuit = study.trials_dataframe()
optuna_resuit.sort_values(by='value', ascending=False).head()
optuna_top5 = optuna_resuit.sort_values(by='value', ascending=False).head()
optuna_top5

,number,value,datetime_start,datetime_complete,duration,params_bagging_fraction,params_bagging_freq,params_colsample_bytree,params_feature_fraction,params_lambda_l1,...,params_learning_rate,params_max_bin,params_max_depth,params_min_child_samples,params_min_child_weight,params_n_estimators,params_num_leaves,params_scale_pos_weight,params_subsample,state
305,305,0.740061,2025-02-26 21:28:32.068625,2025-02-26 21:29:57.379140,0 days 00:01:25.310515,0.520360,2,0.768548,0.819448,2.081722,...,0.005405,552,8,142,2.979813,1692,37,3.574385,0.665180,COMPLETE
342,342,0.740054,2025-02-26 22:21:20.878897,2025-02-26 22:22:41.656732,0 days 00:01:20.777835,0.518023,2,0.875145,0.829536,3.186385,...,0.005241,529,8,136,4.120454,1820,37,3.365979,0.831631,COMPLETE
312,312,0.740048,2025-02-26 21:38:15.751547,2025-02-26 21:39:37.617309,0 days 00:01:21.865762,0.540985,2,0.772574,0.778605,4.763991,...,0.005871,526,8,147,3.098251,1653,35,3.284882,0.963527,COMPLETE
301,301,0.740044,2025-02-26 21:23:50.413315,2025-02-26 21:25:12.159272,0 days 00:01:21.745957,0.516934,2,0.800321,0.802347,0.905354,...,0.005347,550,8,142,3.621465,1685,37,3.821925,0.999113,COMPLETE
338,338,0.740038,2025-02-26 22:16:00.147206,2025-02-26 22:17:21.713531,0 days 00:01:21.566325,0.516469,2,0.852843,0.817527,4.045080,...,0.005004,522,8,140,4.823583,1781,36,3.332666,0.776001,COMPLETE


In [50]:
# train 데이터를 5개로 나누어 교차 검증
under_ratio_df = pd.DataFrame(columns=['fold', "model_num", 'auc'])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for i, index in enumerate(skf.split(X, y)):
    # 데이터 분할
    print(f"Fold {i + 1}")
    X_train, X_val = X.iloc[index[0]], X.iloc[index[1]]
    y_train, y_val = y.iloc[index[0]], y.iloc[index[1]]
    print(Counter(y_train), Counter(y_val))


    param_1 = {'learning_rate': 0.005404665080038044, 'num_leaves': 37, 'max_depth': 8, 'min_child_samples': 142, 'subsample': 0.6651803742719826, 'colsample_bytree': 0.7685478000166595, 'lambda_l1': 2.081721942205924, 'lambda_l2': 1.9845698568428882, 'max_bin': 552, 'n_estimators': 1692, 'scale_pos_weight': 3.574385124249283, 'min_child_weight': 2.9798133779257356, 'feature_fraction': 0.8194483752728194, 'bagging_fraction': 0.5203599421398333, 'bagging_freq': 2}
    param_2 = {'learning_rate': 0.005241044750009647, 'num_leaves': 37, 'max_depth': 8, 'min_child_samples': 136, 'subsample': 0.8316305241255582, 'colsample_bytree': 0.8751454304107738, 'lambda_l1': 3.1863847267496954, 'lambda_l2': 1.9599348709473319, 'max_bin': 529, 'n_estimators': 1820, 'scale_pos_weight': 3.3659790014660134, 'min_child_weight': 4.120453976315193, 'feature_fraction': 0.8295361659356961, 'bagging_fraction': 0.5180226651099835, 'bagging_freq': 2}
    param_3 = {'learning_rate': 0.005870648662761175, 'num_leaves': 35, 'max_depth': 8, 'min_child_samples': 147, 'subsample': 0.963526863489085, 'colsample_bytree': 0.7725737353035547, 'lambda_l1': 4.763990765216757, 'lambda_l2': 1.4307728513570679, 'max_bin': 526, 'n_estimators': 1653, 'scale_pos_weight': 3.2848819311587154, 'min_child_weight': 3.0982512723205846, 'feature_fraction': 0.7786049991397331, 'bagging_fraction': 0.5409850381444363, 'bagging_freq': 2}
    
    param_4 = {'learning_rate': 0.005346878637366447, 'num_leaves': 37, 'max_depth': 8, 'min_child_samples': 142, 'subsample': 0.9991125920471303, 'colsample_bytree': 0.8003207366208486, 'lambda_l1': 0.9053542803984748, 'lambda_l2': 1.381697200704866, 'max_bin': 550, 'n_estimators': 1685, 'scale_pos_weight': 3.8219254462472905, 'min_child_weight': 3.6214646053600323, 'feature_fraction': 0.8023473252909493, 'bagging_fraction': 0.516934474617231, 'bagging_freq': 2}
    param_5 = {'learning_rate': 0.005241044750009647, 'num_leaves': 37, 'max_depth': 8, 'min_child_samples': 136, 'subsample': 0.8316305241255582, 'colsample_bytree': 0.8751454304107738, 'lambda_l1': 3.1863847267496954, 'lambda_l2': 1.9599348709473319, 'max_bin': 529, 'n_estimators': 1820, 'scale_pos_weight': 3.3659790014660134, 'min_child_weight': 4.120453976315193, 'feature_fraction': 0.8295361659356961, 'bagging_fraction': 0.5180226651099835, 'bagging_freq': 2}

    import lightgbm as lgb
    model_1 = lgb.LGBMClassifier(**param_1, random_state=RANDOM_STATE, n_jobs=-1)
    model_2 = lgb.LGBMClassifier(**param_2, random_state=RANDOM_STATE, n_jobs=-1)
    model_3 = lgb.LGBMClassifier(**param_3, random_state=RANDOM_STATE, n_jobs=-1)
    model_4 = lgb.LGBMClassifier(**param_4, random_state=RANDOM_STATE, n_jobs=-1)
    model_5 = lgb.LGBMClassifier(**param_5, random_state=RANDOM_STATE, n_jobs=-1)

    model_1.fit(X_train, y_train)
    model_2.fit(X_train, y_train)
    model_3.fit(X_train, y_train)
    model_4.fit(X_train, y_train)
    model_5.fit(X_train, y_train)

    y_proba_1 = model_1.predict_proba(X_val)[:, 1]
    y_proba_2 = model_2.predict_proba(X_val)[:, 1]
    y_proba_3 = model_3.predict_proba(X_val)[:, 1]
    y_proba_4 = model_4.predict_proba(X_val)[:, 1]
    y_proba_5 = model_5.predict_proba(X_val)[:, 1]

    auc_1 = roc_auc_score(y_val, y_proba_1)
    auc_2 = roc_auc_score(y_val, y_proba_2)
    auc_3 = roc_auc_score(y_val, y_proba_3)
    auc_4 = roc_auc_score(y_val, y_proba_4)
    auc_5 = roc_auc_score(y_val, y_proba_5)

    under_ratio_df.loc[len(under_ratio_df), :] = [i+1, 1, auc_1]
    under_ratio_df.loc[len(under_ratio_df), :] = [i+1, 2, auc_2]
    under_ratio_df.loc[len(under_ratio_df), :] = [i+1, 3, auc_3]
    under_ratio_df.loc[len(under_ratio_df), :] = [i+1, 4, auc_4]
    under_ratio_df.loc[len(under_ratio_df), :] = [i+1, 5, auc_5]

    model_1_pred = model_1.predict_proba(X_val)[:, 1]
    model_2_pred = model_2.predict_proba(X_val)[:, 1]
    model_3_pred = model_3.predict_proba(X_val)[:, 1]
    model_4_pred = model_4.predict_proba(X_val)[:, 1]
    model_5_pred = model_5.predict_proba(X_val)[:, 1]

    # ensemble_pred = (model_1_pred + model_2_pred + model_3_pred + model_4_pred + model_5_pred) / 5
    ensemble_pred = (model_1_pred + model_2_pred + model_3_pred) / 3
    ensemble_auc = roc_auc_score(y_val, ensemble_pred)
    under_ratio_df.loc[len(under_ratio_df), :] = [i+1, 'ensemble', ensemble_auc]

Fold 1
Counter({0: 152098, 1: 52982}) Counter({0: 38025, 1: 13246})
Fold 2
Counter({0: 152098, 1: 52983}) Counter({0: 38025, 1: 13245})
Fold 3
Counter({0: 152098, 1: 52983}) Counter({0: 38025, 1: 13245})
Fold 4
Counter({0: 152099, 1: 52982}) Counter({0: 38024, 1: 13246})
Fold 5
Counter({0: 152099, 1: 52982}) Counter({0: 38024, 1: 13246})


In [51]:
under_ratio_df.groupby(["model_num"]).mean()[['auc']]

,auc
model_num,
1,0.740061
2,0.740054
3,0.740048
4,0.740044
5,0.740054
ensemble,0.740078


In [42]:
under_ratio_df.groupby(["model_num"]).mean()[['auc']]

,auc
model_num,
1,0.740061
2,0.740054
3,0.740048
4,0.740044
5,0.740054
ensemble,0.740077


### 최종 예측

단일

In [41]:
top5_df

,ratio,model,auc
0,default,LGBMClassifier,0.738864
1,2,LGBMClassifier,0.73852
2,ros,LGBMClassifier,0.738255
3,1,LGBMClassifier,0.73778
4,default,XGBClassifier,0.735134


In [45]:
top5_df

,ratio,model,auc,param,tuned_auc
0,default,LGBMClassifier,0.738864,"{'learning_rate': 0.005404665080038044, 'num_l...",0.740061
1,2,LGBMClassifier,0.73852,NaN,NaN
2,ros,LGBMClassifier,0.738255,NaN,NaN
3,1,LGBMClassifier,0.73778,NaN,NaN
4,default,XGBClassifier,0.735134,NaN,NaN


In [46]:
# train_data에서 임신 성공 여부을 제외한 모든 열을 feature로 사용
X = train_encoded.drop(columns=["임신 성공 여부"])
y = train_encoded["임신 성공 여부"]


print('샘플링 진행')
best_ratio = top5_df.loc[0, 'ratio']
print(best_ratio)
if best_ratio == 'default':
    X_sample, y_sample = X, y

elif best_ratio == 'ros':
    ros = RandomOverSampler(random_state=RANDOM_STATE)
    X_ros, y_ros = ros.fit_resample(X, y)
    print("Random Oversampling 클래스 비율:", Counter(y_ros))
    X_sample, y_sample = X_ros, y_ros 
else:
    normal_ratio = best_ratio
    df_normal = train_encoded[train_encoded["임신 성공 여부"] == 0]
    df_abnormal = train_encoded[train_encoded["임신 성공 여부"] == 1]
    print(f"Total: Normal: {len(df_normal)}, AbNormal: {len(df_abnormal)}")

    df_normal = df_normal.sample(n=int(len(df_abnormal) * normal_ratio), replace=False, random_state=RANDOM_STATE)
    df_concat = pd.concat([df_normal, df_abnormal], axis=0).reset_index(drop=True)
    X_sample, y_sample = df_concat.drop(columns=["임신 성공 여부"]), df_concat["임신 성공 여부"]

import xgboost as xgb
import lightgbm as lgb
import catboost as cat

param = top5_df.loc[0, 'param']
if top5_df.loc[0, 'model'] == 'XGBClassifier':
    model = xgb.XGBClassifier(**param, random_state=RANDOM_STATE, n_jobs=-1)
elif top5_df.loc[0, 'model'] == 'LGBMClassifier':
    model = lgb.LGBMClassifier(**param, random_state=RANDOM_STATE, n_jobs=-1)
elif top5_df.loc[0, 'model'] == 'CatBoostClassifier':
    model =  cat.CatBoostClassifier(**param, random_state=RANDOM_STATE, auto_class_weights="Balanced")

model.fit(X_sample, y_sample)
test_proba = model.predict_proba(test_encoded)[:, 1]
test_proba

샘플링 진행
default


array([0.00297803, 0.00247761, 0.38668116, ..., 0.76264853, 0.44227659,
       0.00298511])

In [47]:
sample_submission = pd.read_csv('./data/sample_submission.csv')
sample_submission['probability'] = test_proba
sample_submission.to_csv('./data/튜닝_안나누기_단일_후처리_랜덤수정_350.csv', index=False)

In [48]:
temp_list = ['난자 저장용',  '기증용']
# '기증용, 배아 저장용'
sample_submission.loc[test[test['배아 생성 주요 이유'].isin(temp_list)].index, 'probability'] = 0
sample_submission.loc[test[test['시술 당시 나이'] == -1].index, 'probability'] = 0

In [49]:
sample_submission.to_csv('./data/튜닝_안나누기_단일_후처리_랜덤수정_350.csv', index=False)

In [52]:
# train_data에서 임신 성공 여부을 제외한 모든 열을 feature로 사용
X = train_encoded.drop(columns=["임신 성공 여부"])
y = train_encoded["임신 성공 여부"]

param_1 = {'learning_rate': 0.005404665080038044, 'num_leaves': 37, 'max_depth': 8, 'min_child_samples': 142, 'subsample': 0.6651803742719826, 'colsample_bytree': 0.7685478000166595, 'lambda_l1': 2.081721942205924, 'lambda_l2': 1.9845698568428882, 'max_bin': 552, 'n_estimators': 1692, 'scale_pos_weight': 3.574385124249283, 'min_child_weight': 2.9798133779257356, 'feature_fraction': 0.8194483752728194, 'bagging_fraction': 0.5203599421398333, 'bagging_freq': 2}
param_2 = {'learning_rate': 0.005241044750009647, 'num_leaves': 37, 'max_depth': 8, 'min_child_samples': 136, 'subsample': 0.8316305241255582, 'colsample_bytree': 0.8751454304107738, 'lambda_l1': 3.1863847267496954, 'lambda_l2': 1.9599348709473319, 'max_bin': 529, 'n_estimators': 1820, 'scale_pos_weight': 3.3659790014660134, 'min_child_weight': 4.120453976315193, 'feature_fraction': 0.8295361659356961, 'bagging_fraction': 0.5180226651099835, 'bagging_freq': 2}
param_3 = {'learning_rate': 0.005870648662761175, 'num_leaves': 35, 'max_depth': 8, 'min_child_samples': 147, 'subsample': 0.963526863489085, 'colsample_bytree': 0.7725737353035547, 'lambda_l1': 4.763990765216757, 'lambda_l2': 1.4307728513570679, 'max_bin': 526, 'n_estimators': 1653, 'scale_pos_weight': 3.2848819311587154, 'min_child_weight': 3.0982512723205846, 'feature_fraction': 0.7786049991397331, 'bagging_fraction': 0.5409850381444363, 'bagging_freq': 2}

import lightgbm as lgb
model_1 = lgb.LGBMClassifier(**param_1, random_state=RANDOM_STATE, n_jobs=-1)
model_2 = lgb.LGBMClassifier(**param_2, random_state=RANDOM_STATE, n_jobs=-1)
model_3 = lgb.LGBMClassifier(**param_3, random_state=RANDOM_STATE, n_jobs=-1)

model_1.fit(X, y)
model_2.fit(X, y)
model_3.fit(X, y)

model_1_pred = model_1.predict_proba(test_encoded)[:, 1]
model_2_pred = model_2.predict_proba(test_encoded)[:, 1]
model_3_pred = model_3.predict_proba(test_encoded)[:, 1]

ensemble_pred = (model_1_pred + model_2_pred + model_3_pred) / 3

In [53]:
sample_submission = pd.read_csv('./data/sample_submission.csv')
sample_submission['probability'] = ensemble_pred

# temp_list = ['난자 저장용',  '기증용']
# # '기증용, 배아 저장용'
# sample_submission.loc[test[test['배아 생성 주요 이유'].isin(temp_list)].index, 'probability'] = 0
sample_submission.loc[test[test['시술 당시 나이'] == -1].index, 'probability'] = 0

sample_submission.to_csv('./data/튜닝_안나누기_단일_후처리_랜덤수정_350_앙상블.csv', index=False)